# Example 5: Dry-Run 命令预览

展示 `dry_run()` 功能：正式跑批前先查看会执行哪些命令、消耗多少资源。

两个级别：
- **L1 `plan`**（默认）：零副作用，不创建目录、不执行任何命令
- **L2 `stage`**：真实 staging INP（含 INCLUDE 解析），但求解和提取只记录不执行

> L1 级别不需要 Abaqus 环境即可运行。

In [2]:
import os

from ABQflow import BatchAbaqusProcessor, JobSpec, PreparationSpec, HookSpec, CommandRecord, JobPlan, plan_parallelism, solver_tokens

CWD = os.getcwd()

## 构造 SPEC

In [3]:
# 构造几个不同配置的 spec 来展示 dry_run 的覆盖范围
specs = [
	# 1. 带 preflight 的现有 INP
	JobSpec(
		job_name = "existing_inp_with_check",
		preparation = PreparationSpec(
			kind = "existing_inp",
			source_path = "./examples/ExistingInpBatchJob/cae_file/planar_stress_main.inp",
			options = {"staging_mode": "copy", "resolve_includes": True}
		),
		preflight = "syntaxcheck",
		post_extraction = [
			HookSpec(
				script_path = "./examples/ExistingInpBatchJob/cae_file/get_max_stress_mises.py",
				tasks = [
					{"result_name": "max_stress_mises"},
					{"result_name": "max_displacement"},
				]
			)
		]
	),
	# 2. 模板替换 INP（无 preflight，有 pre-extraction）
	JobSpec(
		job_name = "parameterized_job",
		preparation = PreparationSpec(
			kind = "inp_based",
			source_path = "./examples/SingleParameterizedJob/cae_file/planar_stress_template.inp",
			params = {"youngs_modulus": 200000, "load_magnitude": 2000}
		),
		pre_extraction = [
			HookSpec(
				script_path = "./examples/SingleParameterizedJob/cae_file/get_total_mass.py",
				tasks = [{"result_name": "total_mass"}]
			)
		]
	),
]

print(f"共 {len(specs)} 个 spec")

共 2 个 spec


## L1 Plan 级别：零副作用命令预览

`dry_run('plan')` 不创建任何目录、不执行任何 subprocess。
返回每个 job 将要执行的命令序列、预期路径和资源估算。

In [4]:
processor = BatchAbaqusProcessor(
	batch_data = specs,
	base_output_dir = os.path.join(CWD, "examples/DryRunPreview/planned_output"),
	cpus_per_job = 8,
	duplicate_mode = "overwrite",
)

# L1: 零副作用命令预览
plans = processor.dry_run("plan")

for p in plans:
	print(f"\n{'='*60}")
	print(f"Job: {p.job_name}")
	print(f"Resources: {p.resource_summary['cpus_per_job']} CPUs"
		f" x {p.resource_summary['tokens_per_job']} tokens/job")
	print(f"Paths:")
	for k, v in p.paths.items():
		print(f"  {k}: {v}")
	print(f"Commands ({len(p.commands)} steps):")
	for i, cmd in enumerate(p.commands, 1):
		print(f"  {i}. [{cmd.stage}] {' '.join(cmd.cmd)}")
		print(f"     cwd: {cmd.cwd}")


Job: existing_inp_with_check
Resources: 8 CPUs x 13 tokens/job
Paths:
  inp: c:\SJTU\Projects_Code\24_Abaqus_Pack\examples/DryRunPreview/planned_output\existing_inp_with_check\existing_inp_with_check.inp
  odb: c:\SJTU\Projects_Code\24_Abaqus_Pack\examples/DryRunPreview/planned_output\existing_inp_with_check/existing_inp_with_check.odb
  output_dir: c:\SJTU\Projects_Code\24_Abaqus_Pack\examples/DryRunPreview/planned_output\existing_inp_with_check
Commands (3 steps):
  1. [preflight] abaqus syntaxcheck job=existing_inp_with_check_chk input=c:\SJTU\Projects_Code\24_Abaqus_Pack\examples/DryRunPreview/planned_output\existing_inp_with_check\existing_inp_with_check.inp
     cwd: c:\SJTU\Projects_Code\24_Abaqus_Pack\examples/DryRunPreview/planned_output\existing_inp_with_check
  2. [solver] abaqus job=existing_inp_with_check input=c:\SJTU\Projects_Code\24_Abaqus_Pack\examples/DryRunPreview/planned_output\existing_inp_with_check\existing_inp_with_check.inp cpus=8 interactive
     cwd: c:\SJTU

## 5.4 资源规划预览

`plan_parallelism` 和 `solver_tokens` 可以独立使用，
不需要跑批就能估算并行度和 token 消耗。

In [ ]:
from ABQflow import plan_parallelism, solver_tokens

CPUS = 8
N_JOBS = 10
TOKENS_AVAILABLE = 50  # 你的 license token 上限

tokens_per = solver_tokens(CPUS)
actual_p = plan_parallelism(N_JOBS, CPUS, license_tokens=TOKENS_AVAILABLE)

print(f"Per job: {CPUS} CPUs -> ~{tokens_per} tokens")
print(f"Planned: {N_JOBS} jobs -> actual parallel {actual_p}")
print(f"Peak tokens: {actual_p} x {tokens_per} = {actual_p * tokens_per} / {TOKENS_AVAILABLE} available")

Per job: 8 CPUs -> ~13 tokens
Planned: 10 jobs -> actual parallel 3
Peak tokens: 3 x 13 = 39 / 50 available


## L2 Stage 级别

L2 有文件系统副作用（创建 output_dir、staging INP），
但求解和提取只记录命令不执行。

适用场景：验证 INCLUDE 解析和占位符替换是否正确。

In [6]:
# L2: 真实 preparation + record_only runner
# 取消注释来运行（需要 Abaqus 在 PATH 中）

processor2 = BatchAbaqusProcessor(
	batch_data = specs,
	base_output_dir = os.path.join(CWD, "examples/DryRunPreview/staged_output"),
	cpus_per_job = 4,
	duplicate_mode = "overwrite",
)
stage_plans = processor2.dry_run("stage")
for p in stage_plans:
	print(f"{p.job_name}: {len(p.commands)} commands logged")

existing_inp_with_check: 0 commands logged
parameterized_job: 2 commands logged
